# 08 — Bias, Fairness, and Robustness Evaluation

This notebook evaluates the final cardiac arrest risk model across clinically relevant subgroups and stress-tests model performance under several robustness scenarios.

Outputs:

- `reports/subgroup_performance.csv`
- `reports/subgroup_missingness_tests.csv`
- `reports/robustness_checks.csv`

Subgroups checked: age bands, gender, smoking status, alcohol status, family history of cardiovascular disease, GCS severity category, and triage score group.

In [1]:
from pathlib import Path
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')

# Resolve paths robustly whether the notebook is run from the repository root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'CardiacPatientData.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features import add_clinical_features

DATA_PATH = PROJECT_ROOT / 'data' / 'CardiacPatientData.csv'
MODEL_PATH = PROJECT_ROOT / 'models' / 'best_model.joblib'
REPORTS_DIR = PROJECT_ROOT / 'reports'
SUBGROUP_PERFORMANCE_PATH = REPORTS_DIR / 'subgroup_performance.csv'
SUBGROUP_MISSINGNESS_PATH = REPORTS_DIR / 'subgroup_missingness_tests.csv'
ROBUSTNESS_PATH = REPORTS_DIR / 'robustness_checks.csv'
FINAL_METRICS_PATH = REPORTS_DIR / 'final_model_metrics.csv'

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
POS_LABEL = 1
MIN_CLASS_COUNT_FOR_RANK_METRICS = 2


## Helper functions

The helpers below standardize feature preparation, model training, metrics, subgroup definitions, and missingness comparisons. Metrics that require both outcome classes (AUROC and AUPRC) are reported as missing when a subgroup contains only one class.

In [2]:
RAW_CATEGORICAL_FEATURES = ['Gender', 'Alcoholic', 'Smoke', 'FHCD', 'TriageScore']
ENGINEERED_CATEGORICAL_FEATURES = [
    'age_band',
    'gcs_severity',
    'triage_score_group',
    'hypoxemia_flag',
    'sodium_abnormal_flag',
    'potassium_abnormal_flag',
    'chloride_abnormal_flag',
    'urea_abnormal_flag',
    'creatinine_abnormal_flag',
]
RAW_NUMERIC_FEATURES = ['SBP', 'DBP', 'HR', 'RR', 'BT', 'SpO2', 'Age', 'GCS', 'Na', 'K', 'Cl', 'Urea', 'Ceratinine']


def add_triage_score_group(df):
    '''Create a readable triage score group while preserving missing values.'''
    result = df.copy()
    if 'TriageScore' not in result.columns:
        return result
    triage = result['TriageScore']
    result['triage_score_group'] = np.select(
        [triage.isna(), triage <= 2, triage == 3, triage >= 4],
        ['Missing', '1-2 / highest acuity', '3 / urgent', '4-5 / lower acuity'],
        default='Other / out of range',
    )
    result.loc[triage.isna(), 'triage_score_group'] = np.nan
    return result


def engineer_features(df, include_engineered=True):
    '''Return model features with deterministic clinical features added when requested.'''
    features = df.copy()
    if include_engineered:
        features = add_clinical_features(features)
        features = add_triage_score_group(features)
    return features


def prepare_model_matrix(df, include_engineered=True):
    '''Drop ID and coerce categorical columns to object for consistent one-hot encoding.'''
    X = engineer_features(df, include_engineered=include_engineered).drop(columns=['ID'], errors='ignore')
    candidate_categorical = RAW_CATEGORICAL_FEATURES + (ENGINEERED_CATEGORICAL_FEATURES if include_engineered else [])
    categorical_features = [col for col in candidate_categorical if col in X.columns]
    numeric_features = [col for col in X.columns if col not in categorical_features]
    X_model = X.copy()
    for col in categorical_features:
        X_model[col] = X_model[col].astype('object').where(X_model[col].notna(), np.nan)
    return X_model, numeric_features, categorical_features


def build_preprocessor(numeric_features, categorical_features):
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    return ColumnTransformer(
        transformers=[
            ('numeric', numeric_pipeline, numeric_features),
            ('categorical', categorical_pipeline, categorical_features),
        ],
        sparse_threshold=0.0,
    )


def build_default_final_model(numeric_features, categorical_features, random_state=RANDOM_STATE):
    '''Fallback final-model specification used when models/best_model.joblib is not present.'''
    return Pipeline([
        ('preprocess', build_preprocessor(numeric_features, categorical_features)),
        ('model', RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_leaf=1,
            class_weight=None,
            random_state=random_state,
            n_jobs=-1,
        )),
    ])


def get_holdout_split(df, random_state=RANDOM_STATE):
    y = df['Outcome'].astype(int)
    ids_repeat = bool(df['ID'].duplicated().any())
    if ids_repeat:
        splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=random_state)
        train_idx, test_idx = next(splitter.split(df.drop(columns=['Outcome']), y, groups=df['ID']))
        split_strategy = 'StratifiedGroupKFold by patient ID'
    else:
        train_idx, test_idx = train_test_split(
            np.arange(len(df)),
            test_size=TEST_SIZE,
            stratify=y,
            random_state=random_state,
        )
        split_strategy = 'Stratified train/test split'
    return train_idx, test_idx, split_strategy


def fit_or_load_final_model(train_raw, include_engineered=True):
    X_train, numeric_features, categorical_features = prepare_model_matrix(
        train_raw.drop(columns=['Outcome']),
        include_engineered=include_engineered,
    )
    y_train = train_raw['Outcome'].astype(int)
    if MODEL_PATH.exists() and include_engineered:
        model = joblib.load(MODEL_PATH)
        source = str(MODEL_PATH.relative_to(PROJECT_ROOT))
    else:
        model = build_default_final_model(numeric_features, categorical_features, random_state=RANDOM_STATE)
        model.fit(X_train, y_train)
        source = 'fallback RandomForestClassifier trained in this notebook'
    return model, source


def score_model(model, raw_df, include_engineered=True):
    X, _, _ = prepare_model_matrix(raw_df.drop(columns=['Outcome']), include_engineered=include_engineered)
    y = raw_df['Outcome'].astype(int)
    y_score = model.predict_proba(X)[:, 1]
    return X, y, y_score


def threshold_metrics(y_true, y_score, threshold):
    y_true = pd.Series(y_true).astype(int)
    y_pred = (np.asarray(y_score) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'sensitivity': recall_score(y_true, y_pred, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'precision_ppv': precision_score(y_true, y_pred, zero_division=0),
        'npv': tn / (tn + fn) if (tn + fn) else np.nan,
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'true_negative': int(tn),
        'false_positive': int(fp),
        'false_negative': int(fn),
        'true_positive': int(tp),
    }


def discrimination_metrics(y_true, y_score):
    y_true = pd.Series(y_true).astype(int)
    class_counts = y_true.value_counts()
    if y_true.nunique() < 2 or class_counts.min() < MIN_CLASS_COUNT_FOR_RANK_METRICS:
        return {'auroc': np.nan, 'auprc': np.nan}
    return {
        'auroc': roc_auc_score(y_true, y_score),
        'auprc': average_precision_score(y_true, y_score),
    }


def all_metrics(y_true, y_score, threshold):
    metrics = {
        'sample_size': int(len(y_true)),
        'outcome_rate': float(pd.Series(y_true).mean()) if len(y_true) else np.nan,
    }
    metrics.update(discrimination_metrics(y_true, y_score))
    metrics.update(threshold_metrics(y_true, y_score, threshold))
    return metrics


def safe_chi_square_pvalue(group_values, missing_indicator):
    table = pd.crosstab(group_values.fillna('Missing'), missing_indicator.astype(int))
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    try:
        return float(chi2_contingency(table)[1])
    except ValueError:
        return np.nan


## Load data, recreate the held-out test split, and score the final model

This mirrors prior notebooks by keeping repeated patient IDs together in the hold-out split. If a saved model artifact is available, it is used directly. If not, the notebook trains the same Random Forest final-model fallback used for robustness comparisons so that the bias analysis remains executable from a fresh clone.

In [3]:
df = pd.read_csv(DATA_PATH)
train_idx, test_idx, split_strategy = get_holdout_split(df, random_state=RANDOM_STATE)
train_raw = df.iloc[train_idx].copy()
test_raw = df.iloc[test_idx].copy()

model, model_source = fit_or_load_final_model(train_raw, include_engineered=True)
X_test_model, y_test, y_score = score_model(model, test_raw, include_engineered=True)

if FINAL_METRICS_PATH.exists():
    threshold = float(pd.read_csv(FINAL_METRICS_PATH).loc[0, 'recommended_threshold'])
    threshold_source = str(FINAL_METRICS_PATH.relative_to(PROJECT_ROOT))
else:
    threshold = 0.50
    threshold_source = 'default threshold because saved final metrics/model artifact was not available'

overall_metrics = all_metrics(y_test, y_score, threshold)

print(f'Data: {DATA_PATH.relative_to(PROJECT_ROOT)}')
print(f'Model source: {model_source}')
print(f'Split strategy: {split_strategy}')
print(f'Train rows: {len(train_raw):,}; test rows: {len(test_raw):,}')
print(f'Test event rate: {y_test.mean():.3f}')
print(f'Threshold: {threshold:.2f} ({threshold_source})')
print(pd.Series(overall_metrics))


Data: data/CardiacPatientData.csv
Model source: fallback RandomForestClassifier trained in this notebook
Split strategy: StratifiedGroupKFold by patient ID
Train rows: 4,746; test rows: 1,160
Test event rate: 0.873
Threshold: 0.75 (reports/final_model_metrics.csv)
sample_size       1160.000000
outcome_rate         0.873276
auroc                0.799773
auprc                0.957306
sensitivity          0.886476
specificity          0.442177
precision_ppv        0.916327
npv                  0.361111
f1                   0.901154
true_negative       65.000000
false_positive      82.000000
false_negative     115.000000
true_positive      898.000000
dtype: float64


## Subgroup performance

For each subgroup level, the table reports sample size, outcome rate, AUROC, AUPRC, sensitivity, specificity, PPV, NPV, and F1. Threshold-dependent metrics use the final-model recommended threshold when available, otherwise 0.50.

In [4]:
subgroup_frame = engineer_features(test_raw.drop(columns=['Outcome']), include_engineered=True)
subgroup_frame['Outcome'] = test_raw['Outcome'].astype(int).values
subgroup_frame['predicted_risk'] = y_score

# Human-readable labels. Keep binary values explicit because the source data dictionary may define coding separately.
subgroup_definitions = {
    'Age bands': 'age_band',
    'Gender': 'Gender',
    'Smoking status': 'Smoke',
    'Alcohol status': 'Alcoholic',
    'Family history of cardiovascular disease': 'FHCD',
    'GCS severity category': 'gcs_severity',
    'Triage score group': 'triage_score_group',
}

rows = []
for subgroup_name, column in subgroup_definitions.items():
    values = subgroup_frame[column].astype('object').where(subgroup_frame[column].notna(), 'Missing')
    for level in sorted(values.unique(), key=lambda x: str(x)):
        mask = values == level
        row = {
            'subgroup': subgroup_name,
            'subgroup_variable': column,
            'subgroup_level': str(level),
        }
        row.update(all_metrics(subgroup_frame.loc[mask, 'Outcome'], subgroup_frame.loc[mask, 'predicted_risk'], threshold))
        rows.append(row)

subgroup_performance = pd.DataFrame(rows)
metric_order = [
    'subgroup', 'subgroup_variable', 'subgroup_level', 'sample_size', 'outcome_rate',
    'auroc', 'auprc', 'sensitivity', 'specificity', 'precision_ppv', 'npv', 'f1',
    'true_negative', 'false_positive', 'false_negative', 'true_positive',
]
subgroup_performance = subgroup_performance[metric_order]
subgroup_performance.to_csv(SUBGROUP_PERFORMANCE_PATH, index=False)

print(f'Saved subgroup performance to: {SUBGROUP_PERFORMANCE_PATH.relative_to(PROJECT_ROOT)}')
display(subgroup_performance)


Saved subgroup performance to: reports/subgroup_performance.csv


,subgroup,subgroup_variable,subgroup_level,sample_size,outcome_rate,auroc,auprc,sensitivity,specificity,precision_ppv,npv,f1,true_negative,false_positive,false_negative,true_positive
0,Age bands,age_band,18-39,124,1.000000,NaN,NaN,1.000000,NaN,1.000000,NaN,1.000000,0,0,0,124
1,Age bands,age_band,40-64,479,0.918580,0.652652,0.949704,1.000000,0.025641,0.920502,1.000000,0.958606,1,38,0,440
2,Age bands,age_band,65-79,487,0.778234,0.883856,0.966994,0.881266,0.592593,0.883598,0.587156,0.882431,64,44,45,334
3,Age bands,age_band,80+,70,1.000000,NaN,NaN,0.000000,NaN,0.000000,0.000000,0.000000,0,0,70,0
4,Gender,Gender,0,507,0.804734,0.918920,0.962205,0.960784,0.606061,0.909513,0.789474,0.934446,60,39,16,392
5,Gender,Gender,1,653,0.926493,0.651154,0.963695,0.836364,0.104167,0.921676,0.048077,0.876950,5,43,99,506
6,Smoking status,Smoke,0.0,512,0.818359,0.893358,0.976716,0.856802,0.634409,0.913486,0.495798,0.884236,59,34,60,359
7,Smoking status,Smoke,1.0,473,0.898520,0.658946,0.951528,0.870588,0.104167,0.895884,0.083333,0.883055,5,43,55,370
8,Smoking status,Smoke,Missing,175,0.965714,0.276134,0.943453,1.000000,0.166667,0.971264,1.000000,0.985423,1,5,0,169
9,Alcohol status,Alcoholic,0.0,579,0.756477,0.757918,0.897857,0.840183,0.453901,0.826966,0.477612,0.833522,64,77,70,368


## Missingness by subgroup

The table below evaluates whether predictor missingness varies across each subgroup definition. The `any_predictor_missing_rate` values are subgroup-level descriptive rates. The chi-square tests screen each raw predictor's missingness indicator against the subgroup variable, then report the minimum p-value and variables with p < 0.05. These tests are exploratory and unadjusted for multiple comparisons.

In [5]:
missingness_base = engineer_features(test_raw.drop(columns=['Outcome']), include_engineered=True)
raw_predictor_cols = [col for col in df.columns if col not in ['ID', 'Outcome']]
missingness_base['any_predictor_missing'] = missingness_base[raw_predictor_cols].isna().any(axis=1)

missingness_rows = []
for subgroup_name, column in subgroup_definitions.items():
    group_values = missingness_base[column].astype('object').where(missingness_base[column].notna(), 'Missing')
    pvalue_rows = []
    for predictor in raw_predictor_cols:
        pvalue_rows.append({
            'predictor': predictor,
            'missingness_p_value': safe_chi_square_pvalue(group_values, missingness_base[predictor].isna()),
        })
    pvalues = pd.DataFrame(pvalue_rows).dropna(subset=['missingness_p_value'])
    significant_predictors = pvalues.loc[pvalues['missingness_p_value'] < 0.05, 'predictor'].tolist()
    min_p = pvalues['missingness_p_value'].min() if len(pvalues) else np.nan

    for level in sorted(group_values.unique(), key=lambda x: str(x)):
        mask = group_values == level
        missingness_rows.append({
            'subgroup': subgroup_name,
            'subgroup_variable': column,
            'subgroup_level': str(level),
            'sample_size': int(mask.sum()),
            'any_predictor_missing_count': int(missingness_base.loc[mask, 'any_predictor_missing'].sum()),
            'any_predictor_missing_rate': float(missingness_base.loc[mask, 'any_predictor_missing'].mean()),
            'minimum_predictor_missingness_p_value': min_p,
            'n_predictors_with_missingness_p_lt_0_05': len(significant_predictors),
            'predictors_with_missingness_p_lt_0_05': ', '.join(significant_predictors),
        })

subgroup_missingness = pd.DataFrame(missingness_rows)
subgroup_missingness.to_csv(SUBGROUP_MISSINGNESS_PATH, index=False)

# Add missingness descriptors to the primary subgroup performance report as requested.
subgroup_performance_with_missingness = subgroup_performance.merge(
    subgroup_missingness,
    on=['subgroup', 'subgroup_variable', 'subgroup_level', 'sample_size'],
    how='left',
)
subgroup_performance_with_missingness.to_csv(SUBGROUP_PERFORMANCE_PATH, index=False)

print(f'Saved missingness tests to: {SUBGROUP_MISSINGNESS_PATH.relative_to(PROJECT_ROOT)}')
print(f'Updated subgroup performance with missingness columns: {SUBGROUP_PERFORMANCE_PATH.relative_to(PROJECT_ROOT)}')
display(subgroup_missingness)


Saved missingness tests to: reports/subgroup_missingness_tests.csv
Updated subgroup performance with missingness columns: reports/subgroup_performance.csv


,subgroup,subgroup_variable,subgroup_level,sample_size,any_predictor_missing_count,any_predictor_missing_rate,minimum_predictor_missingness_p_value,n_predictors_with_missingness_p_lt_0_05,predictors_with_missingness_p_lt_0_05
0,Age bands,age_band,18-39,124,123,0.991935,5.781893e-172,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
1,Age bands,age_band,40-64,479,110,0.229645,5.781893e-172,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
2,Age bands,age_band,65-79,487,217,0.445585,5.781893e-172,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
3,Age bands,age_band,80+,70,0,0.000000,5.781893e-172,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
4,Gender,Gender,0,507,304,0.599606,1.054658e-58,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
5,Gender,Gender,1,653,146,0.223583,1.054658e-58,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
6,Smoking status,Smoke,0.0,512,167,0.326172,1.285880e-252,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
7,Smoking status,Smoke,1.0,473,108,0.228330,1.285880e-252,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
8,Smoking status,Smoke,Missing,175,175,1.000000,1.285880e-252,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."
9,Alcohol status,Alcoholic,0.0,579,214,0.369603,1.285880e-252,9,"Na, K, Cl, Urea, Ceratinine, Alcoholic, Smoke,..."


## Robustness checks

Robustness checks rerun a comparable Random Forest modeling pipeline under:

1. Alternative leakage-safe train/test splits.
2. Outlier handling through train-set IQR removal and train-derived winsorization.
3. Feature-set comparison with and without engineered clinical features.

These are not intended to replace the final model selection workflow; they identify whether conclusions are sensitive to reasonable analysis choices.

In [6]:
def fit_evaluate_pipeline(train_df, test_df, scenario, random_state=RANDOM_STATE, include_engineered=True):
    X_train, numeric_features, categorical_features = prepare_model_matrix(
        train_df.drop(columns=['Outcome']),
        include_engineered=include_engineered,
    )
    X_test, _, _ = prepare_model_matrix(
        test_df.drop(columns=['Outcome']),
        include_engineered=include_engineered,
    )
    y_train = train_df['Outcome'].astype(int)
    y_eval = test_df['Outcome'].astype(int)
    estimator = build_default_final_model(numeric_features, categorical_features, random_state=random_state)
    estimator.fit(X_train, y_train)
    eval_score = estimator.predict_proba(X_test)[:, 1]
    row = {
        'scenario': scenario,
        'random_state': random_state,
        'include_engineered_features': include_engineered,
        'train_rows': len(train_df),
        'test_rows': len(test_df),
        'test_event_rate': float(y_eval.mean()),
        'threshold': threshold,
    }
    row.update(all_metrics(y_eval, eval_score, threshold))
    return row


def remove_train_outliers_iqr(train_df, numeric_cols, multiplier=1.5):
    train_clean = train_df.copy()
    mask = pd.Series(True, index=train_clean.index)
    for col in numeric_cols:
        if col not in train_clean.columns:
            continue
        q1 = train_clean[col].quantile(0.25)
        q3 = train_clean[col].quantile(0.75)
        iqr = q3 - q1
        if pd.isna(iqr) or iqr == 0:
            continue
        lower = q1 - multiplier * iqr
        upper = q3 + multiplier * iqr
        mask &= train_clean[col].isna() | train_clean[col].between(lower, upper)
    return train_clean.loc[mask].copy()


def winsorize_from_train(train_df, test_df, numeric_cols, lower_q=0.01, upper_q=0.99):
    train_w = train_df.copy()
    test_w = test_df.copy()
    for col in numeric_cols:
        if col not in train_w.columns:
            continue
        lower = train_w[col].quantile(lower_q)
        upper = train_w[col].quantile(upper_q)
        train_w[col] = train_w[col].clip(lower, upper)
        test_w[col] = test_w[col].clip(lower, upper)
    return train_w, test_w

robustness_rows = []

# Baseline comparable pipeline on the primary split.
robustness_rows.append(fit_evaluate_pipeline(train_raw, test_raw, 'primary split / engineered features', RANDOM_STATE, True))

# Alternative splits.
for seed in [7, 2024, 2026]:
    alt_train_idx, alt_test_idx, _ = get_holdout_split(df, random_state=seed)
    alt_train = df.iloc[alt_train_idx].copy()
    alt_test = df.iloc[alt_test_idx].copy()
    robustness_rows.append(fit_evaluate_pipeline(alt_train, alt_test, f'alternative split seed {seed}', seed, True))

# Outlier removal / winsorization on the primary split.
outlier_removed_train = remove_train_outliers_iqr(train_raw, RAW_NUMERIC_FEATURES, multiplier=1.5)
robustness_rows.append(fit_evaluate_pipeline(outlier_removed_train, test_raw, 'train IQR outlier removal', RANDOM_STATE, True))

winsor_train, winsor_test = winsorize_from_train(train_raw, test_raw, RAW_NUMERIC_FEATURES, lower_q=0.01, upper_q=0.99)
robustness_rows.append(fit_evaluate_pipeline(winsor_train, winsor_test, 'train-derived 1st/99th percentile winsorization', RANDOM_STATE, True))

# With vs without engineered features.
robustness_rows.append(fit_evaluate_pipeline(train_raw, test_raw, 'primary split / raw features only', RANDOM_STATE, False))

robustness_df = pd.DataFrame(robustness_rows)
robustness_df.to_csv(ROBUSTNESS_PATH, index=False)

print(f'Saved robustness checks to: {ROBUSTNESS_PATH.relative_to(PROJECT_ROOT)}')
display(robustness_df)


Saved robustness checks to: reports/robustness_checks.csv


,scenario,random_state,include_engineered_features,train_rows,test_rows,test_event_rate,threshold,sample_size,outcome_rate,auroc,auprc,sensitivity,specificity,precision_ppv,npv,f1,true_negative,false_positive,false_negative,true_positive
0,primary split / engineered features,42,True,4746,1160,0.873276,0.75,1160,0.873276,0.799773,0.957306,0.886476,0.442177,0.916327,0.361111,0.901154,65,82,115,898
1,alternative split seed 7,7,True,4746,1160,0.873276,0.75,1160,0.873276,0.904836,0.981598,0.866732,0.877551,0.979911,0.488636,0.919853,129,18,135,878
2,alternative split seed 2024,2024,True,4746,1160,0.873276,0.75,1160,0.873276,0.949644,0.989688,0.946693,0.857143,0.978571,0.700000,0.962368,126,21,54,959
3,alternative split seed 2026,2026,True,4746,1160,0.873276,0.75,1160,0.873276,0.814305,0.962296,0.950642,0.340136,0.908491,0.500000,0.929088,50,97,50,963
4,train IQR outlier removal,42,True,3768,1160,0.873276,0.75,1160,0.873276,0.858426,0.973599,0.900296,0.523810,0.928717,0.432584,0.914286,77,70,101,912
5,train-derived 1st/99th percentile winsorization,42,True,4746,1160,0.873276,0.75,1160,0.873276,0.799343,0.960229,0.865745,0.442177,0.914494,0.323383,0.889452,65,82,136,877
6,primary split / raw features only,42,False,4746,1160,0.873276,0.75,1160,0.873276,0.831158,0.970454,0.905232,0.428571,0.916084,0.396226,0.910626,63,84,96,917


## Bias, fairness, and robustness discussion

### How to interpret subgroup results

- **Small subgroup sizes:** AUROC and AUPRC are unstable when a subgroup has few records or very few non-events/events. This notebook leaves AUROC/AUPRC blank when a subgroup has only one outcome class.
- **Outcome-rate differences:** Large differences in outcome rate can change PPV and NPV even if AUROC is similar. PPV generally increases and NPV generally decreases in higher-prevalence subgroups.
- **Threshold effects:** Sensitivity, specificity, PPV, NPV, and F1 are evaluated at a single global threshold. A global threshold may not yield equal operating characteristics across subgroups if baseline risk, measurement patterns, or calibration differ.
- **Missingness:** Missing data can encode care processes, documentation quality, or disease severity. If missingness differs by subgroup, performance gaps may partly reflect how tests and vital signs were collected rather than true clinical risk.

### Possible bias sources

- **Selection bias:** The dataset may include only patients captured by this study's inclusion criteria, hospital workflows, or emergency-care documentation systems. Patients treated elsewhere, transferred, discharged quickly, or missing key records may be underrepresented.
- **Measurement bias:** Vitals, laboratory values, triage scores, smoking/alcohol history, and GCS can be measured or documented differently across clinicians, time periods, and patient groups. Differential measurement frequency can also produce subgroup-specific missingness.
- **Repeated patient records:** Repeated `ID` values suggest multiple rows may come from the same patient. The split keeps IDs together, but subgroup estimates can still be influenced by high-utilization patients contributing multiple observations.
- **Single-site data:** If the data come from one site or health system, the model may learn local practice patterns, triage norms, patient demographics, or equipment/documentation conventions that do not generalize to other settings.
- **Label quality:** `Outcome` may be affected by coding accuracy, timing of cardiac arrest ascertainment, censoring, delayed events, or ambiguous clinical definitions. Label noise can reduce apparent performance and may be worse for some subgroups.

### Practical recommendations

- Review subgroups with low sensitivity or low NPV before clinical use, because missed high-risk patients are the main safety concern for screening.
- Validate externally on data from additional sites and time periods.
- Consider patient-level analysis if repeated records reflect longitudinal measurements, including a sensitivity analysis that samples one record per patient.
- Audit missingness mechanisms and documentation practices, especially for predictors with significant subgroup missingness differences.
- Assess calibration by subgroup before using risk probabilities for decision thresholds or resource allocation.